<a href="https://colab.research.google.com/github/dlkt101101/AMATH445/blob/main/3D_Reconstruction_LoFTR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install kornia
# !pip install kornia-rs
# !pip install kornia_moons
# !pip install opencv-python --upgrade
# !pip install depth-anything-v2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 51.2 MB/s eta 0:00:00


In [2]:
import os, cv2
import kornia as K
import kornia.feature as KF
import matplotlib.pyplot as plt
import numpy as np
import torch
import open3d as o3d

from kornia_moons.viz import draw_LAF_matches
from PIL import Image
from cv2 import findEssentialMat, recoverPose, RANSAC
from hloc import extract_features, match_features, reconstruction, pairs_from_sequential
from pathlib import Path
from hloc.utils.parsers import parse_retrieval

# Import data

Replace the below with another function to get data since all is on github now

In [ ]:
cap = cv2.VideoCapture("tunnel.mp4")
os.makedirs("frames", exist_ok=True)
i = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if i % 5 == 0:  # Sample every 5th frame
        cv2.imwrite(f"frames/frame_{i:05d}.png", frame)
    i += 1
cap.release()

# Feature Matching

We can test between pre-trained weights on `indoor` or `outdoor` data.

In [3]:
matcher = KF.LoFTR(pretrained='indoor')

Downloading: "http://cmp.felk.cvut.cz/~mishkdmy/models/loftr_indoor.ckpt" to /root/.cache/torch/hub/checkpoints/loftr_indoor.ckpt


100%|██████████| 44.2M/44.2M [00:00<00:00, 139MB/s]


below function matches 2 consecutive frames

In [ ]:
def load_gray_tensor(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return torch.tensor(img / 255., dtype=torch.float32)[None, None]

img0 = load_gray_tensor("frames/frame_00000.png")
img1 = load_gray_tensor("frames/frame_00005.png")

with torch.no_grad():
    input_dict = {"image0": img0, "image1": img1}
    correspondences = matcher(input_dict)

kpts0 = correspondences["keypoints0"].numpy()  # (N, 2)
kpts1 = correspondences["keypoints1"].numpy()  # (N, 2)
confidence = correspondences["confidence"].numpy()

# Pose Estimation

In [ ]:
# Camera intrinsics K (from calibration or EXIF)
K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]])

E, mask = findEssentialMat(kpts0, kpts1, K, method=RANSAC, prob=0.999, threshold=1.0)
_, R, t, mask = recoverPose(E, kpts0, kpts1, K)

# R = rotation matrix, t = translation vector (direction of camera movement)

In [ ]:
images = Path("frames/")
outputs = Path("outputs/")

# Use LoFTR as the matcher
matcher_conf = match_features.confs["loftr"]

# For sequential video, use sequential pairs (not exhaustive)
pairs = outputs / "pairs.txt"
pairs_from_sequential.main(images, pairs, overlap=5)

# Run matching + COLMAP reconstruction
features = extract_features.main(...)
matches = match_features.main(matcher_conf, pairs, features=features)
model = reconstruction.main(sfm_dir=outputs / "sfm", image_dir=images, pairs=pairs, features=features, matches=matches)

# Depth Estimation

In [ ]:
positions = [np.zeros(3)]
for R, t in pose_pairs:
    new_pos = positions[-1] + R.T @ t.squeeze()
    positions.append(new_pos)

tunnel_depth = [np.linalg.norm(p) for p in positions]

 # 3D Model Reconstruction

In [ ]:
# Load depth maps + camera poses and unproject to 3D
pcd = o3d.geometry.PointCloud()
for depth_map, K, pose in zip(depth_maps, intrinsics, camera_poses):
    pts = unproject(depth_map, K, pose)
    pcd.points.extend(pts)

o3d.io.write_point_cloud("tunnel_reconstruction.ply", pcd)